# 平均 IQ 与类型化读取

这是独立的合成沙盒，不连接设备。无需完成其他课程。先从新 kernel 顺序运行，再做文末的小修改。
启动入口已准备环境与服务；重复打开继续当前练习，重置会创建新的起点。

In [ ]:
import scopecat as sc

session = sc.notebook()
session

In [ ]:
import my_experiment.teaching as experiments
import numpy as np
from my_experiment.setup import open_parameters

params = open_parameters(session)

`src/my_experiment/teaching.py` 已包含逐 shot 和平均 IQ 两个实验。平均函数是普通 NumPy 代码，`@sc.compute` 让实验定义中的调用加入计算图。

`result_types.py` 中的 `type IQ = Annotated[complex, sc.Unit("ratio")]` 同时用于计算返回值和下面的读取字段。先运行，不需要手工拼装实验。

In [ ]:
from dataclasses import dataclass

from my_experiment.result_types import IQ


@dataclass(frozen=True)
class MeanRow:
    amplitude: sc.Quantity
    iq: IQ


scan = np.linspace(0, 0.8, 7)
raw = (
    session.prepare(
        experiments.teaching_rabi().sweep(amplitude=scan), parameters=params
    )
    .run()
    .wait(timeout=120)
    .result()
)
mean = (
    session.prepare(experiments.mean_rabi().sweep(amplitude=scan), parameters=params)
    .run()
    .wait(timeout=120)
    .result()
)
rows = mean.result().rows_as(MeanRow)
np.testing.assert_allclose(
    [row.iq for row in rows],
    np.asarray(raw.measurements()["iq"].require_values()).mean(axis=1),
)
assert len(rows) == 7
print(rows[0])

保存身份后可以重启 kernel；只运行检查内核、下面的导入与读取单元，即可读取既有记录，无需重新采集。

In [ ]:
number = session.run_number(mean)
print("本项目运行编号:", number)
session.history()

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class MeanRow:
    amplitude: sc.Quantity
    iq: IQ


with sc.open_project().authoring() as reader:
    reopened = reader.run(number).result().rows_as(MeanRow)
    assert len(reopened) == 7
    print("重开:", reopened[0])

小修改：在 `teaching.py` 的 `mean_iq` 中改变一个计算因子；保存后直接重跑创建新请求的单元；Notebook 默认工作区自动采用已保存源码。对照断言会指出科学含义发生变化，先解释差异再修改预期。
平均结果没有 shot 分布，不应直接传给依赖 shot 误差估计的拟合函数。

重启 kernel 后，重新连接并执行 `session.history()`，从本地时间和实验名称找到目标编号，然后用 `session.run(编号)` 读取。编号是本项目内的稳定编号，不是列表位置；跨项目引用仍保留完整 run ID。这里没有额外 JSON 文件，也不以“最新一次运行”替代确切身份。

本地 `response.py` 和 `mean_iq` 都使用 `@sc.compute`。`shot_array(shots)` 为本次请求声明固定 shot 轴；装饰器自动传入 shots 参数；`IQ` 声明标量单位。实验返回 dict 选择要记录的字段，不需要额外输出 dataclass；`MeanRow` 是可选的读取校验。